# ICCD753 Recuperación de Información — Examen Final
**Tema:** Diseño e Implementación de un Sistema de Recuperación de Información (RAG)

**Estudiante:** [Mateo Coronado]

---
### Enlace de Despliegue en la Nube
**URL Streamlit Cloud:** [https://examri-hgffynhtki64tforybeqie.streamlit.app/]

---

## Visión General del Sistema

El sistema implementa un pipeline completo de **Retrieval-Augmented Generation (RAG)** sobre resúmenes de artículos científicos de arXiv. La arquitectura se divide en dos fases operativas:

| Fase | Momento | Usa API de Google | Descripción |
|------|---------|:-----------------:|-------------|
| **Offline** (indexación) | Una sola vez |  No | Descarga datos, genera embeddings localmente, almacena vectores en ChromaDB |
| **Online** (consulta) | Cada pregunta del usuario |  Sí (Gemini Flash) | Recupera documentos similares, genera respuesta con el LLM |

### Estructura de Archivos
```
Exam_RI/
├── src/
│   ├── data_processing.py    # (A) Descarga y preparación del corpus
│   ├── embeddings.py          # (B) Generación de embeddings locales
│   ├── vector_db.py           # (C) Almacenamiento vectorial (ChromaDB)
│   ├── index_corpus.py        # Script que orquesta B + C
│   ├── retrieval.py           # (D) Recuperación semántica
│   ├── generation.py          # (E) Generación con LLM (Gemini)
│   ├── app.py                 # (F/G) Interfaz web Streamlit
│   ├── evaluate.py            # (I) Evaluación cuantitativa
│   ├── evaluation_metrics.py  # Funciones de métricas (P@k, R@k, NDCG@k)
│   └── generate_qrels.py      # Generador de Ground Truth
├── data/
│   ├── processed/corpus.json  # Corpus procesado (2000 papers)
│   ├── chroma_db/             # Base de datos vectorial persistente
│   └── evaluation/            # Resultados de evaluación
├── Proyecto_Final_RAG.ipynb   # Este notebook
├── requirements.txt
├── README.md
└── .env                       # GOOGLE_API_KEY (solo local)
```

### Tecnologías Utilizadas

| Librería | Uso en el proyecto |
|----------|-------------------|
| `sentence-transformers` | Embeddings locales (all-MiniLM-L6-v2, 384 dims) |
| `chromadb` | Base de datos vectorial persistente con métrica coseno |
| `langchain` + `langchain-google-genai` | Cadena LCEL: PromptTemplate → Gemini 2.5 Flash |
| `streamlit` | Interfaz web tipo chat con `st.chat_input`, `st.expander` |
| `kagglehub` | Descarga automática del dataset de arXiv |
| `pandas` | Lectura y procesamiento del CSV |

---
## A. Preparación del Corpus — `src/data_processing.py`

**Comando:** `python src/data_processing.py`

| Función | Qué hace |
|---------|----------|
| `main()` | Orquesta todo el flujo de descarga y procesamiento |
| `setup_directories()` | Crea la carpeta `data/processed/` si no existe |
| `process_arxiv_data(df, limit)` | Itera sobre las filas del DataFrame, concatena título y abstract, y construye la lista de documentos |

**Flujo detallado:**
1. Llama a `kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")` para descargar el CSV (~52k papers).
2. Lee el CSV con `pd.read_csv()` y detecta automáticamente las columnas (`title`, `abstract`, `terms`, `url`).
3. Para cada paper (limitado a **2000**), concatena título y resumen en el formato: `Title: <título> \n Abstract: <resumen>`.
4. Guarda el resultado en `data/processed/corpus.json`.

In [ ]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f"Total papers en el dataset: {len(df)}")
print(f"Columnas: {list(df.columns)}")
df.head(2)

---
## B. Representación mediante Embeddings — `src/embeddings.py`

Los embeddings se generan **100% de forma local** usando la librería `sentence-transformers`. No se consume ninguna cuota de la API de Google en este paso.

| Clase | Método | Qué hace |
|-------|--------|----------|
| `TextEmbedder` | `__init__(model_name)` | Descarga e inicializa el modelo local `all-MiniLM-L6-v2` de HuggingFace mediante `SentenceTransformer(model_name)`. Se descarga solo la primera vez (~80MB). |
| `TextEmbedder` | `embed_text(texts)` | Recibe un string o lista de strings, ejecuta `self.model.encode(texts).tolist()` y devuelve una lista de vectores de **384 dimensiones** cada uno. |

In [ ]:
from src.embeddings import TextEmbedder

embedder = TextEmbedder()
test_embedding = embedder.embed_text("Graph Neural Networks")
print(f"Dimensión del embedding: {len(test_embedding[0])}")
print(f"Primeros 5 valores: {test_embedding[0][:5]}")

---
## C. Almacenamiento y Búsqueda Vectorial — `src/vector_db.py`

Se utiliza **ChromaDB** para almacenar de manera persistente los documentos y sus correspondientes vectores semánticos.

| Clase | Método | Qué hace |
|-------|--------|----------|
| `VectorDB` | `__init__(db_dir, collection_name)` | Crea un cliente persistente en `data/chroma_db/`. Crea o recupera la colección `arxiv_corpus` con métrica **coseno** (`hnsw:space: cosine`). |
| `VectorDB` | `index_documents(ids, embeddings, documents, metadatas)` | Inserta (o actualiza con `upsert`) documentos y vectores en lotes de 100. |
| `VectorDB` | `search(query_embedding, top_k)` | Ejecuta `collection.query()` con el vector de consulta y retorna los `top_k` documentos más cercanos. |

### Indexación — `src/index_corpus.py`

**Comando:** `python src/index_corpus.py`

Este script **orquesta** los módulos B y C:
1. Carga el archivo `data/processed/corpus.json`.
2. Instancia `TextEmbedder()` (modelo local) y `VectorDB()` (ChromaDB).
3. Recorre el corpus en **lotes de 100 documentos**: genera los 100 vectores con `embedder.embed_text()` y los almacena con `db.index_documents()`.
4. Muestra progreso con la barra `tqdm`.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="data/chroma_db")
collection = client.get_or_create_collection(
    name="arxiv_corpus",
    metadata={"hnsw:space": "cosine"}
)
print(f"Documentos indexados actualmente: {collection.count()}")

---
## D. Recuperación Semántica — `src/retrieval.py`

| Clase | Método | Qué hace |
|-------|--------|----------|
| `TextRetriever` | `__init__()` | Instancia `TextEmbedder()` y `VectorDB()` |
| `TextRetriever` | `retrieve(query_text, top_k=5)` | 1) Genera el embedding del query con `embedder.embed_text()`. 2) Busca en ChromaDB con `db.search()`. 3) Formatea los resultados con `_format_results()`. |
| `TextRetriever` | `_format_results(results)` | Transforma la respuesta cruda de ChromaDB en una lista de diccionarios con campos: `id`, `score`, `text`, `title`, `topics`, `url`. |

**Cadena de llamadas:**
```
query_text → embedder.embed_text() → db.search() → _format_results() → [lista de evidencias]
```

In [ ]:
from src.retrieval import TextRetriever

retriever = TextRetriever()
results = retriever.retrieve("What are the main applications of Graph Neural Networks?", top_k=3)
for r in results:
    print(f"Score: {r['score']:.4f} | Title: {r['title']}")

---
## E. Generación Aumentada por Recuperación (RAG) — `src/generation.py`

Este módulo toma los documentos recuperados por el Retriever, construye un contexto textual y lo envía junto con la pregunta del usuario al LLM (Gemini 2.5 Flash) para generar una respuesta conversacional.

| Clase | Método | Qué hace | Cadena de llamadas |
|-------|--------|----------|--------------------|
| `RAGGenerator` | `__init__(model_name, temperature)` | Inicializa el cliente de Gemini vía LangChain. Define el `PromptTemplate` con 2 variables: `context`, `question`. Conecta ambos en una cadena LCEL: `self.chain = self.prompt \| self.llm`. | `ChatGoogleGenerativeAI()` → `PromptTemplate()` → operador pipe `\|` |
| `RAGGenerator` | `format_context(retrieved_docs)` | Toma la lista de diccionarios de evidencias y los concatena en un solo string con formato legible para el LLM: `Documento N: - Título: ... - Contenido: ...` | — (concatenación de strings) |
| `RAGGenerator` | `generate_response(query, retrieved_docs)` | Orquesta todo: formatea el contexto, invoca la cadena LCEL y devuelve el texto de la respuesta. Si falla la API, captura la excepción y devuelve un mensaje de error amigable. | `format_context()` → `chain.invoke({context, question})` → `response.content` |

> **Nota:** La temperatura está fijada en `0.0` para maximizar la determinación y minimizar las alucinaciones del modelo.

### El Prompt Template (lo que Gemini realmente recibe):

```
Eres un asistente académico experto. Utiliza ÚNICAMENTE la siguiente información
recuperada de resúmenes de artículos científicos (contexto) para responder a la
consulta del usuario.

Reglas Estrictas:
1. Responde basándote exclusivamente en la información provista en el contexto.
2. Si la respuesta a la consulta del usuario NO se encuentra en el contexto, DEBES
   indicar explícitamente que el corpus no contiene información suficiente para
   responder la consulta. Bajo ninguna circunstancia inventes o deduzcas información
   externa.
3. Si usas información de un artículo, cita su título sutilmente para respaldar
   tu respuesta.

Contexto Recuperado:
{context}

Consulta del Usuario: {question}
```

In [ ]:
from src.generation import RAGGenerator

generator = RAGGenerator()
answer = generator.generate_response(
    "What are the main applications of Graph Neural Networks?", 
    results
)
print("Respuesta Generada:\n", answer)

---
## F. Presentación de Evidencias e Interfaz Web — `src/app.py`

La interfaz web se construyó con Streamlit y funciona como un chat conversacional.

| Función / Bloque | Qué hace |
|-------------------|----------|
| Carga de API Key | Busca `GOOGLE_API_KEY` primero en `.env` (local), luego en `st.secrets` (Streamlit Cloud). Si no existe, detiene la app con `st.stop()`. |
| `load_pipeline()` | Decorada con `@st.cache_resource` para que solo se cargue una vez. Instancia `TextRetriever()` y `RAGGenerator()`. |
| `render_evidences(evidences)` | Renderiza cada evidencia dentro de un `st.expander()` mostrando tópicos, URL y resumen del paper. |
| Flujo del Chat | Usa `st.chat_input()` → `retriever.retrieve(prompt, top_k=3)` → `generator.generate_response(prompt, evidences)` → muestra respuesta + evidencias. |

**Cadena completa de una consulta del usuario:**

```
[Usuario escribe pregunta]
    → st.chat_input()
    → retriever.retrieve(query, top_k=3)
        → embedder.embed_text(query)       ← LOCAL
        → db.search(vector, 3)             ← LOCAL (ChromaDB)
    → generator.generate_response(query, evidences)
        → format_context(evidences)
        → chain.invoke({context, question}) ← API GEMINI
    → st.markdown(respuesta)
    → render_evidences(evidencias)          ← st.expander()
```

Para ejecutar la interfaz:
```bash
streamlit run src/app.py
```

---
## I. Evaluación del Sistema y de la Generación

### Evaluación Cualitativa (Juicio subjetivo)

A continuación, se documenta el juicio subjetivo sobre el desempeño del sistema RAG tras realizar pruebas manuales con distintas consultas:

- **Corrección de la respuesta:** El modelo Gemini-2.5-Flash produce respuestas altamente correctas semántica y gramaticalmente. Al establecer la temperatura en `0.0`, el modelo se restringe a responder de forma determinista y precisa.
- **Relevancia con respecto a la consulta:** Al usar embeddings locales (`all-MiniLM-L6-v2`) y la métrica de similitud del coseno en ChromaDB, el sistema recupera satisfactoriamente los artículos que abordan el núcleo semántico de la pregunta del usuario.
- **Fidelidad respecto de las evidencias recuperadas:** El *prompt estricto* diseñado obliga al LLM a no utilizar conocimiento previo, forzando a que las afirmaciones generadas se correspondan uno a uno con el texto de las evidencias presentadas en pantalla.
- **Capacidad para integrar información de varios documentos:** El modelo es capaz de leer el contexto concatenado de los fragmentos Top-K, sintetizando ideas (por ejemplo, si dos papers hablan de grafos, agrupa las aplicaciones mencionadas en ambos en una única lista coherente).
- **Capacidad para reconocer cuando el corpus no contiene información suficiente:** Gracias a las instrucciones condicionadas en el prompt (regla 2), si se le pregunta por una receta de cocina o un tema ajeno a arXiv, el sistema responde explícitamente: *"el corpus no contiene información suficiente para responder la consulta"*, cumpliendo con éxito la mitigación de alucinaciones.

### Evaluación Cuantitativa — `src/evaluate.py`

El sistema cuenta con los scripts `src/generate_qrels.py` y `src/evaluate.py` que calculan de forma programática el desempeño del sistema utilizando las métricas estándar de Recuperación de Información.

**Comandos:**
```bash
python src/generate_qrels.py   # Genera Ground Truth (5 queries aleatorias)
python src/evaluate.py          # Calcula P@k, R@k, NDCG@k
```

| Función | Fórmula |
|---------|----------|
| `precision_at_k(retrieved, relevant, k)` | `|relevantes ∩ top_k| / k` |
| `recall_at_k(retrieved, relevant, k)` | `|relevantes ∩ top_k| / |relevantes|` |
| `ndcg_at_k(retrieved, relevant, k)` | `DCG@k / IDCG@k` (normalizado) |

**Resultados obtenidos:**

| k | Precision | Recall | NDCG |
|---|-----------|--------|------|
| 1 | 0.8000 | 0.8000 | 0.8000 |
| 3 | 0.3333 | 1.0000 | 0.9262 |
| 5 | 0.2000 | 1.0000 | 0.9262 |
| 10 | 0.1000 | 1.0000 | 0.9262 |

> Un **Recall@3 = 1.0** y un **NDCG@3 = 0.93** demuestran que el sistema recupera exitosamente el documento correcto dentro del Top-3 en el 100% de los casos evaluados.